# Source Scenes for Static Layers over Australia

- The following notebook is used to search for the unique set of scenes that can be used to produced burst static layer products for the Australian region.
- Unfortunately, the OPERA JPL team did not create static layers for the Antarctic Region. Therefore, the method used to identify a complete set of scene's is different than the approach used for Australia
- The approach for Antarctica is;

1. Load in the burst-db created with the [make_upload_burst_db.sh](../scripts/make_upload_burst_db.sh) script
2. Filter for burst ID's covering Antarctica
3. For each burst id in the database, query the ODATA burst API for the parent product name : https://documentation.dataspace.copernicus.eu/APIs/Sentinel-1%20SLC%20Burst.html

# Imports

In [ ]:
import sqlite3
import asf_search
import pandas as pd
import pandas as pd
import geopandas as gpd
from shapely import segmentize
from shapely.geometry import box, Polygon, shape
from tqdm import tqdm
from datetime import datetime, timedelta
import re
import pyproj
import random

## Settings

In [ ]:
# Only want to find scenes before this date, set to NONE if not to be considered
MAX_SCENE_DATE = datetime(2026, 1, 1)

# Functions

In [ ]:
def query_database(query, params=None):
    """Connects to the SQLite database and executes a query."""
    try:
        # Connect to the database
        
        # Execute the query
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        # Fetch results
        results = cursor.fetchall()

        # Close the connection
        conn.close()

        return results

    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        return None

# Function to list tables
def list_tables():
    """Retrieves all table names from the SQLite database."""
    query = "SELECT name FROM sqlite_master WHERE type='table';"
    tables = query_database(query)
    return [table[0] for table in tables] if tables else []

def get_scene_for_burstID(fullBurstID, end=MAX_SCENE_DATE):

    # remove t from fullBurstID if there
    fullBurstID = fullBurstID[1:] if fullBurstID[0] == 't' else fullBurstID
    
    results = asf_search.search(
        platform=[asf_search.PLATFORM.SENTINEL1], 
        maxResults=1, 
        processingLevel='BURST',
        fullBurstID=fullBurstID
    )
    
    if len(results) == 0:
        return None
    # get the scene from the BURST URL
    burst_url = results[0].properties['url']
    scene_id = burst_url.split('/')[3]

    return scene_id

def parse_scene_file_dates(scene_id: str) -> tuple[datetime, datetime]:
    """Extracts start_date and end_date from the given scene ID.

    Parameters
    ----------
    scene_id : str
        Sentinel-1 scene ID
        e.g. S1A_EW_GRDM_1SDH_20220612T120348_20220612T120452_043629_053582_0F6

    Returns
    -------
    tuple[datetime, datetime]
        A tuple containing the start and stop date for the scene as datetimes
        e.g. (datetime(2022,06,12,12,3,48), datetime(2022,06,12,12,4,52))

    Raises
    ------
    ValueError
        Did not find a match to the expected date pattern of start_date followed by end_date in the scene ID
    """
    # Regex pattern to match the dates
    pattern = r"(?P<start_date>\d{8}T\d{6})_" r"(?P<stop_date>\d{8}T\d{6})_"

    match = re.search(pattern, scene_id)

    if not match:
        raise ValueError("The input string does not match the expected format.")

    start_date = datetime.strptime(match.group("start_date"), "%Y%m%dT%H%M%S")
    stop_date = datetime.strptime(match.group("stop_date"), "%Y%m%dT%H%M%S")

    return (start_date, stop_date)

def get_burst_ids_for_scene(scene: str) -> list[str]:
    """Get the list of burst_ids corresponding to a scene

    Parameters
    ----------
    scene : str
        the scene id. e.g. S1A_IW_SLC__1SSH_20220101T124744_20220101T124814_041267_04E7A2_1DAD

    Returns
    -------
    list[str]
        List of burst ids. e.g. ['070_149822_IW3','070_149822_IW2' ....]
    """

    if scene is None:
        return []

    st, et = parse_scene_file_dates(scene)
    
    results = asf_search.search(
            platform=[asf_search.PLATFORM.SENTINEL1], 
            maxResults=100, 
            processingLevel='BURST',
            start=st-timedelta(seconds=2),
            end=et+timedelta(seconds=2),
        )
    
    burst_ids = [r.properties['burst']['fullBurstID'] for r in results if scene in r.properties['url']]
    burst_ids = [f't{b.lower()}' for b in burst_ids]
    return burst_ids

def transform_polygon(
    geometry: Polygon, src_crs: int, dst_crs: int, always_xy: bool = True
):
    src_crs = pyproj.CRS(f"EPSG:{src_crs}")
    dst_crs = pyproj.CRS(f"EPSG:{dst_crs}")
    transformer = pyproj.Transformer.from_crs(src_crs, dst_crs, always_xy=always_xy)
    # Transform the polygon's coordinates
    if isinstance(geometry, Polygon):
        # Transform exterior
        exterior_coords = [
            transformer.transform(x, y) for x, y in geometry.exterior.coords
        ]
        # Transform interiors (holes)
        interiors_coords = [
            [transformer.transform(x, y) for x, y in interior.coords]
            for interior in geometry.interiors
        ]
        # Create the transformed polygon
        return Polygon(exterior_coords, interiors_coords)

    # Handle other geometry types as needed
    raise ValueError("Only Polygon geometries are supported for transformation.")


def segmentize_reproject_geometry(
    geometry:  tuple[float | int, float | int, float | int, float | int],
    src_crs: int,
    ref_crs: int,
    segment_length: float = 0.1,
) -> tuple:
    """_summary_

    Parameters
    ----------
    bounds : BoundingBox | tuple[float | int, float | int, float | int, float | int],
        Bounds to adjust.
    src_crs : int
        Source EPSG. e.g. 4326
    ref_crs : int
        Reference crs to create the true bbox. i.e. 3031 in southern
        hemisphere and 3995 in northern (polar stereographic)
    segment_length : float, optional
        distance between generation points along the bounding box sides in
        src_crs. e.g. 0.1 degrees in lat/lon, by default 0.1

    Returns
    -------
    BoundingBox
        A polygon bounding box expanded to the true min max
    """
    segmentized_geometry = segmentize(geometry, max_segment_length=segment_length)
    transformed_geometry = transform_polygon(segmentized_geometry, src_crs, ref_crs)
    return transformed_geometry


## Download the burst-db and Antarctic shapefiles for the Area of Interest Covering Bursts

In [ ]:
# ! wget https://data.dev.dea.ga.gov.au/projects/s1_nrb/burst_db/0.9.0/opera-burst-bbox-only.sqlite3
# ! wget https://data.dev.dea.ga.gov.au/projects/s1_nrb/production_aois/antarctica_aoi_excl_antimeridian_polygon.geojson
# ! wget https://data.dev.dea.ga.gov.au/projects/s1_nrb/historical_scene_coverage_aois/merged_antartctic_aoi_sentinel_1_iw_grd_scenes_footprint_2014_to_2023.geojson

## Load the burst-db into a pandas DataFrame

In [ ]:
db_path = 'opera-burst-bbox-only.sqlite3'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
query = "SELECT name FROM sqlite_master WHERE type='table';"
cursor.execute(query)
results = cursor.fetchall()
results

In [ ]:
table_name = "burst_id_map"
query = f"PRAGMA table_info({table_name});"
cursor.execute(query)
columns = cursor.fetchall()
columns

In [ ]:
table_name = "burst_id_map"
query = f"SELECT * from ({table_name}) limit 10;"
cursor.execute(query)
burst_info = cursor.fetchall()
burst_info

In [ ]:
df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
print(f"burst count in db : {len(df)}")
df.head(2)

## Convert all geometries to 3031

In [ ]:
target_crs = 3031
projected_dfs = []

# Group by EPSG to minimize transformation overhead
for epsg_code, group in tqdm(df.groupby("epsg")):
    # Create bounding boxes
    geom = [box(xmin, ymin, xmax, ymax) for xmin, ymin, xmax, ymax in zip(group.xmin, group.ymin, group.xmax, group.ymax)]
    
    gdf = gpd.GeoDataFrame(group.copy(), geometry=geom, crs=f"EPSG:{epsg_code}")
    gdf_proj = gdf.to_crs(epsg=target_crs)
    projected_dfs.append(gdf_proj)

# Combine all into one GeoDataFrame
gdf_3031 = pd.concat(projected_dfs, ignore_index=True)

In [ ]:
gdf_3031.head(2)

# Burst ID's intersecting with shapefile
- Limit the search space to locations where historical IW scenes have covered.

In [ ]:
import geopandas as gpd
from shapely import segmentize

# read in the Antarctic shapefile used by cophub
#aoi_gdf = gpd.read_file('antarctica_aoi_excl_antimeridian_polygon.geojson')
# segment_length=0.1
# aoi_gdf.geometry = aoi_gdf.geometry.apply(lambda x : segmentize(x, max_segment_length=segment_length))
# aoi_gdf_3031 = aoi_gdf.to_crs(epsg=3031)
# read in the shapefile where historical IW scenes have covered.
aoi_gdf_3031 = gpd.read_file('merged_antartctic_aoi_sentinel_1_iw_grd_scenes_footprint_2014_to_2023.geojson')
aoi_gdf_3031.dissolve().iloc[0].geometry

In [ ]:
gdf_intersecting = gdf_3031.sjoin(
    aoi_gdf_3031,
    predicate="intersects",
    how="inner"
)
gdf_intersecting = gdf_intersecting.set_index("burst_id_jpl")
gdf_3031 = gdf_3031.set_index("burst_id_jpl")

In [ ]:
print(f'Number of bursts intersecting geometry: {len(gdf_intersecting)}')

In [ ]:
gdf_intersecting.dissolve().plot()

In [ ]:
#get_scene_for_burstID('t089_190162_iw3')

In [ ]:
# clear some memory
df = None

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_scene_for_burstID_parallel(search_bursts, max_workers=None):
    """
    Parallel lookup of scene IDs for a list of burst IDs.

    Parameters
    ----------
    search_bursts : list[str]
        Burst IDs (e.g. 't076_162583_iw2')
    max_workers : int | None
        Number of threads (defaults to len(search_bursts))

    Returns
    -------
    list[str | None]
        Scene IDs in the same order as search_bursts.
        None if lookup failed.
    """

    if not search_bursts:
        return []

    max_workers = max_workers or min(8, len(search_bursts))
    results = [None] * len(search_bursts)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(get_scene_for_burstID, burst): i
            for i, burst in enumerate(search_bursts)
        }

        for future in as_completed(future_to_index):
            idx = future_to_index[future]
            try:
                results[idx] = future.result()
            except Exception as exc:
                # Fail soft: log and continue
                print(f"Scene lookup failed for {search_bursts[idx]}: {exc}")
                results[idx] = None

    return results

from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

def get_burst_ids_for_scene_parallel(
    scene_ids: List[str],
    max_workers: int | None = None
) -> List[List[str]]:
    """
    Parallel lookup of burst IDs for multiple Sentinel‑1 scenes.

    Parameters
    ----------
    scene_ids : list[str]
        List of Sentinel‑1 scene IDs
    max_workers : int | None
        Number of threads (defaults to min(8, len(scene_ids)))

    Returns
    -------
    list[list[str]]
        A list of burst‑ID lists, aligned with scene_ids.
        If a scene fails, an empty list is returned.
    """

    if not scene_ids:
        return []

    max_workers = max_workers or min(8, len(scene_ids))
    results: List[List[str]] = [[] for _ in scene_ids]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(get_burst_ids_for_scene, scene): i
            for i, scene in enumerate(scene_ids)
        }

        for future in as_completed(future_to_index):
            idx = future_to_index[future]
            try:
                results[idx] = future.result()
            except Exception as exc:
                # Fail soft: log and continue
                print(f"Burst lookup failed for scene {scene_ids[idx]}: {exc}")
                results[idx] = []

    return results

In [ ]:
scene_to_burst_list = {}
burst_info = []
remaining_bursts_to_search = set(gdf_intersecting.index.values)
searched_bursts = set()
n_burst_ids = len(remaining_bursts_to_search)

N_PARALLEL = 10
LOG_EVERY = N_PARALLEL * 1
BREAK_AFTER = None
cnt = 0

while True:
    search_bursts = random.sample(list(remaining_bursts_to_search), N_PARALLEL)
    cnt += len(search_bursts)
    scenes = get_scene_for_burstID_parallel(search_bursts, max_workers=N_PARALLEL)
    scenes_burst_ids = get_burst_ids_for_scene_parallel(scenes, max_workers=N_PARALLEL)
    for i,scene in enumerate(scenes):
        burst_id = search_bursts[i]
        scene_burst_ids = scenes_burst_ids[i]
        if scene is None:
            searched_bursts = searched_bursts | set([burst_id])
            remaining_bursts_to_search = remaining_bursts_to_search - set([burst_id])
        else:
            if burst_id in searched_bursts:
                continue
            scene_dt = datetime.strptime(scene.split("_")[6], "%Y%m%dT%H%M%S")
            new_bursts = set(scene_burst_ids) - searched_bursts # only get bew bursts
            searched_bursts = searched_bursts | new_bursts # updated what we have searched for
            remaining_bursts_to_search = remaining_bursts_to_search - set(scene_burst_ids) # subract scene bursts from remaining to look for
            scene_to_burst_list[scene] = list(new_bursts)
            for b in new_bursts:
                burst_geom = burst_geom = gdf_3031.loc[b].geometry
                burst_info.append(
                    {
                        "burst_id": b,
                        "scene_id": scene,
                        "scene_dt": scene_dt,
                        "geometry": burst_geom
                    }
                )

    if cnt % LOG_EVERY == 0:
        print(f"{len(searched_bursts)} of {n_burst_ids} ({100*(len(searched_bursts)/n_burst_ids):.3f}) bursts searched. {len(scene_to_burst_list)} scenes found. {len(burst_info)} bursts with data.")
    if BREAK_AFTER:
        if cnt > BREAK_AFTER:
            break
    if (len(remaining_bursts_to_search) == 0) or (searched_bursts >= n_burst_ids):
        print(f'Search finished, no bursts remaining!')


In [ ]:
gdf = gpd.GeoDataFrame(
    burst_info,
    geometry="geometry",
    crs="EPSG:3031",  # CMR geometries are lon/lat
)
gdf.plot()

In [ ]:
# write list
with open("antarctica_static_layer_source_scene_ids.txt", "w") as f:
    for scene in scene_to_burst_list.keys():
        f.write(f"{scene}\n")

In [ ]:
# write geojson
gdf.to_file("antarctica_static_layer_source_scenes.geojson", driver="GeoJSON")  # readable, portable

## Read in files and analyse

In [ ]:
bursts_3031 = gpd.read_file('antarctica_static_layer_source_scenes.geojson')
bursts_3031.plot()

In [ ]:
df = pd.read_csv("antarctica_static_layer_source_scene_ids.txt", header=None)
df = df.rename(columns={0:"scene_id"})
df['scene_dt'] = df['scene_id'].apply(lambda x : datetime.strptime(x.split("_")[6], "%Y%m%dT%H%M%S"))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure datetime
df['scene_dt'] = pd.to_datetime(df['scene_dt'])

# Group by scene_id to remove duplicate bursts in same scene
scene_counts = df.groupby('scene_id').first().reset_index()

# Set datetime index for resampling
scene_counts.set_index('scene_dt', inplace=True)

# Resample monthly
monthly_counts = scene_counts['scene_id'].resample('M').count()

# Plot
fig, ax = plt.subplots(figsize=(10,5))
monthly_counts.plot(kind='bar', width=0.8, ax=ax)

# Major ticks: every 12 months → show year
ax.set_xticks(range(0, len(monthly_counts), 12))
ax.set_xticklabels([d.strftime('%Y') for d in monthly_counts.index[::12]], rotation=45)

ax.set_xlabel("Date")
ax.set_ylabel("Number of Scenes per Month")
ax.set_title(f"Monthly Count of Input Scenes Used to Create Static Layers over Antarctica (total = {len(scene_counts)})")
ax.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
